<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-09/NB09_uncertainty.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 09: Uncertainty, Calibration and Distribution Shift

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-09/lab.html>.

## Overview

A safe system must know when it does not know. This week studies calibration, the estimation of predictive uncertainty, the detection of inputs unlike the training data and conformal prediction, which turns any model into one with a distribution-free coverage guarantee [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to compute and interpret the expected calibration error, to apply temperature scaling, to explain deep ensembles and Monte Carlo dropout, to classify types of dataset shift and use a maximum softmax probability detector, and to construct split conformal prediction sets and verify their coverage.

## Why confidence matters for safety

Many safety mechanisms depend on a model's confidence. A diagnostic assistant should defer to a clinician when unsure, and an autonomous vehicle should slow down when its perception is uncertain. Such mechanisms work only if confidence is meaningful. A model is calibrated when, among predictions made with confidence p, the fraction that is correct is also p. The expected calibration error (ECE) measures the gap: Predictions are grouped into confidence bins, and the absolute differences between accuracy and mean confidence are averaged with weights proportional to the bin sizes.

Guo and colleagues found that modern deep networks, although more accurate than older ones, are often overconfident, and they linked the change to greater depth and width, batch normalisation and weaker weight decay [1]. They also found that temperature scaling, which divides all logits by a single constant T fitted on held-out data, is a simple and effective remedy. Because dividing by T does not change which class has the largest logit, temperature scaling leaves accuracy unchanged while improving confidence estimates. Figure 9.1 shows the effect on a small network.

![Figure 9.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-09/figures/w09_fig1.png)

*Figure 9.1. Reliability diagrams on a held-out test set before and after temperature scaling for a neural network trained on noisy digits [1].*

## Estimating uncertainty

Two kinds of uncertainty are usually distinguished. Aleatoric uncertainty comes from noise in the data and cannot be reduced by more data of the same kind. Epistemic uncertainty comes from limited knowledge of the model and shrinks with more data. Safety failures often involve epistemic uncertainty, when a model meets inputs unlike its training data.

Gal and Ghahramani showed that keeping dropout active at test time and averaging several stochastic forward passes approximates Bayesian inference, which yields uncertainty estimates from networks that were trained with dropout anyway [4]. Lakshminarayanan, Pritzel and Blundell proposed deep ensembles: Several networks are trained independently from different random initialisations, and their predictions are averaged [5]. Ensembles are simple, parallel and among the strongest practical baselines for both calibration and the detection of unfamiliar inputs.

## Distribution shift and out-of-distribution detection

Deployed models rarely see data from exactly the training distribution. The collection edited by Quiñonero-Candela and colleagues describes several forms of dataset shift [6]. Covariate shift changes the input distribution while the relation between inputs and labels stays fixed. Prior probability shift changes the frequency of classes. Concept shift changes the relation itself. Amodei and colleagues listed robustness to distributional shift among the concrete problems of AI safety [7].

Hendrycks and Gimpel proposed a baseline for detecting misclassified and out-of-distribution inputs: the maximum softmax probability, which tends to be lower for such inputs [2]. The baseline is useful and easy to deploy, but it has a clear limit. Inputs that resemble training data in the wrong way can receive high confidence. The notebook shows this: The detector separates random noise from digits reasonably well, yet it fails on rotated digits, which the network classifies with confidence.

## Distribution-free guarantees: conformal prediction

Conformal prediction, developed by Vovk, Gammerman and Shafer, wraps any model in a procedure that outputs prediction sets with a guaranteed coverage rate [8]. Angelopoulos and Bates gave an accessible introduction to its split form [3]. A nonconformity score is computed for each example of a held-out calibration set, for instance one minus the probability assigned to the true class. The empirical quantile of these scores at level ceil((n + 1)(1 - alpha)) / n defines a threshold, and the prediction set for a new input contains every class whose score falls below it. If calibration and test data are exchangeable, the set contains the true label with probability at least 1 - alpha.

The guarantee is marginal, averaged over inputs, rather than conditional on each input, and it breaks when the test distribution differs from the calibration distribution. Set size is informative in itself: A large set signals a hard or unfamiliar input, which is exactly when a safety mechanism should ask for help.

> **Pause and reflect.** A triage model outputs the set {pneumonia, bronchitis} for one patient and {pneumonia} for another. How should a clinical workflow treat the two cases differently?

# Hands-on lab

The notebook trains a small neural network on noisy digits, measures and repairs its calibration, compares it with a deep ensemble, builds conformal prediction sets and tests the maximum softmax probability detector on two kinds of unfamiliar inputs [1, 2, 3, 5].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A network on noisy digits

The data are split into training, calibration and test sets. The calibration set is used both for temperature scaling and for conformal prediction.

In [ ]:
import warnings
from scipy.optimize import minimize_scalar
from sklearn.datasets import load_digits
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")

rng = np.random.default_rng(SEED)
d = load_digits()
X = d.data / 16.0 + rng.normal(0, 0.25, d.data.shape)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, d.target, test_size=0.5, random_state=0)
X_cal, X_te, y_cal, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=1)
mlp = MLPClassifier(hidden_layer_sizes=(128,), alpha=1e-5, max_iter=600, random_state=0).fit(X_tr, y_tr)

def logits(m, X):
    h = np.maximum(0, X @ m.coefs_[0] + m.intercepts_[0])
    return h @ m.coefs_[1] + m.intercepts_[1]

def softmax(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def ece(p, y, bins=10):
    conf, pred, total = p.max(1), p.argmax(1), 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any():
            total += m.mean() * abs(np.mean(pred[m] == y[m]) - conf[m].mean())
    return total

z_cal, z_te = logits(mlp, X_cal), logits(mlp, X_te)
print("test accuracy:", np.mean(z_te.argmax(1) == y_te).round(3), " ECE:", round(ece(softmax(z_te), y_te), 4))

## 2. Temperature scaling

In [ ]:
nll = lambda T: -np.mean(np.log(softmax(z_cal / T)[np.arange(len(y_cal)), y_cal] + 1e-12))
T = minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x
print(f"fitted temperature T = {T:.3f}")
print("ECE after scaling:", round(ece(softmax(z_te / T), y_te), 4), " accuracy unchanged:", np.mean((z_te / T).argmax(1) == y_te).round(3))

## 3. A deep ensemble

In [ ]:
members = [MLPClassifier(hidden_layer_sizes=(128,), alpha=1e-5, max_iter=600, random_state=s).fit(X_tr, y_tr) for s in range(5)]
p_ens = np.mean([m.predict_proba(X_te) for m in members], axis=0)
print("ensemble accuracy:", np.mean(p_ens.argmax(1) == y_te).round(3), " ensemble ECE:", round(ece(p_ens, y_te), 4))

## 4. Split conformal prediction

### Your turn, exercise 1

The cell below builds prediction sets at alpha = 0.1. Compute the empirical coverage: the fraction of test examples whose true label is in their prediction set.

In [ ]:
alpha = 0.1
p_cal, p_te = softmax(z_cal / T), softmax(z_te / T)
scores = 1 - p_cal[np.arange(len(y_cal)), y_cal]
n = len(scores)
q_hat = np.quantile(scores, np.ceil((n + 1) * (1 - alpha)) / n, method="higher")
sets = (1 - p_te) <= q_hat          # boolean matrix: test example x class
print(f"threshold {q_hat:.3f}, mean set size {sets.sum(1).mean():.2f}")
coverage = None  # your computation

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _coverage_reference():
    return float(np.mean(sets[np.arange(len(y_te)), y_te]))

In [ ]:
check("Exercise 1", coverage, _coverage_reference())
sizes = sets.sum(1)
plt.hist(sizes, bins=np.arange(0.5, 10.5), rwidth=0.85, color="#1D5B78")
plt.xlabel("prediction set size"); plt.ylabel("count"); plt.title("Harder inputs receive larger sets"); plt.show()

## 5. Detecting unfamiliar inputs

Two kinds of unfamiliar inputs are compared with the test digits: uniform noise and the same test digits rotated by 90 degrees.

In [ ]:
X_rot = np.array([np.rot90(x.reshape(8, 8)).ravel() for x in X_te])
X_noise = rng.uniform(0, 1, X_te.shape)
msp_in = p_te.max(1)
msp_rot = softmax(logits(mlp, X_rot) / T).max(1)
msp_noise = softmax(logits(mlp, X_noise) / T).max(1)

def auroc(inside, outside):
    return roc_auc_score(np.r_[np.ones(len(inside)), np.zeros(len(outside))], np.r_[inside, outside])

print("AUROC, digits versus rotated digits:", round(auroc(msp_in, msp_rot), 3))
plt.hist(msp_in, bins=30, alpha=0.6, label="test digits"); plt.hist(msp_rot, bins=30, alpha=0.6, label="rotated digits")
plt.hist(msp_noise, bins=30, alpha=0.6, label="noise"); plt.legend(); plt.xlabel("maximum softmax probability"); plt.show()

### Your turn, exercise 2

Compute the AUROC of the detector for test digits against noise images using the `auroc` helper. Then explain in two sentences why rotated digits are harder to detect than noise.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _auroc_noise_reference():
    return auroc(msp_in, msp_noise)

In [ ]:
auroc_noise = None  # your computation
check("Exercise 2", auroc_noise, _auroc_noise_reference())

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-09/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Describe a decision in your field where a model should abstain. Which signal from this week would you use to trigger abstention?
2. Conformal guarantees break under distribution shift. How would you monitor for shift after deployment?
3. Was the model in the notebook more dangerous before or after temperature scaling? Argue both sides briefly.

## Weekly task and submission

Evaluate the uncertainty of a classifier from your field or the notebook model. Report accuracy, ECE before and after temperature scaling, conformal coverage and mean set size at alpha = 0.1, and an out-of-distribution test of your choice. Discuss in about 300 words how the results would change a deployment decision.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Reporting uncertainty in clinical AI.** Examine how a sample of clinical AI studies reports calibration and uncertainty. Propose a short reporting checklist based on the methods of this week [1, 3].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks. In *Proceedings of the 34th International Conference on Machine Learning, PMLR 70* (pp. 1321-1330).

[2] Hendrycks, D., & Gimpel, K. (2017). A baseline for detecting misclassified and out-of-distribution examples in neural networks. In *International Conference on Learning Representations (ICLR)*. <https://arxiv.org/abs/1610.02136>

[3] Angelopoulos, A. N., & Bates, S. (2023). Conformal prediction: A gentle introduction. *Foundations and Trends in Machine Learning*, 16(4), 494-591. <https://doi.org/10.1561/2200000101>

[4] Gal, Y., & Ghahramani, Z. (2016). Dropout as a Bayesian approximation: Representing model uncertainty in deep learning. In *Proceedings of the 33rd International Conference on Machine Learning, PMLR 48* (pp. 1050-1059).

[5] Lakshminarayanan, B., Pritzel, A., & Blundell, C. (2017). Simple and scalable predictive uncertainty estimation using deep ensembles. In *Advances in Neural Information Processing Systems 30*.

[6] Quiñonero-Candela, J., Sugiyama, M., Schwaighofer, A., & Lawrence, N. D. (Eds.) (2009). *Dataset Shift in Machine Learning*. MIT Press.

[7] Amodei, D., Olah, C., Steinhardt, J., Christiano, P., Schulman, J., & Mané, D. (2016). Concrete problems in AI safety. arXiv preprint arXiv:1606.06565. <https://arxiv.org/abs/1606.06565>

[8] Vovk, V., Gammerman, A., & Shafer, G. (2005). *Algorithmic Learning in a Random World*. Springer.